# Relation Extraction Pipeline

Extracts **entity→entity edges** from cached chunks. All edges are typed `RELATES_TO`
and carry a plain-English `relation_description` — one sentence from the source explaining
how the two entities are connected. No standardized vocabulary, no closed enum.

**Why `RELATES_TO` + description instead of typed predicates:**
- PPR only needs an edge to walk — it does not need a predicate label to traverse the graph.
- Natural-language descriptions preserve the actual semantics better than a closed vocab
  that forces every relation into `hasPart` / `usesTool` / `other`.
- Downstream: you can cluster descriptions or embed them to recover semantic groupings
  without having baked in a vocabulary decision at extraction time.

**Pipeline (3 stages):**
1. **Stage 1** — LLM finds entity pairs and writes one sentence per pair.
2. **Stage 2** — Object resolution: object name → canonical entity ID via `mention_to_canonical`.
3. **Stage 3** — Span verification + confidence scoring.

**Inputs:** `entity_cache.json`, `resolved_entities.json`  
**Output:** `relations_cache.json` — `RelationEdge` records staged for Neo4j write.

## Setup & imports

In [48]:
import json
import re
import sys
import hashlib
from pathlib import Path
from typing import NamedTuple

from dotenv import load_dotenv
from openai import OpenAI
from pydantic import BaseModel

load_dotenv()
client = OpenAI()

ENTITY_CACHE    = Path("entity_cache.json")
RESOLVED_CACHE  = Path("resolved_entities.json")
RELATIONS_CACHE = Path("relations_cache.json")

EXTRACTOR_VERSION = "relation-extraction-v1/gpt-4.1-mini"
EXTRACT_MODEL     = "gpt-4.1-mini"

for p in (ENTITY_CACHE, RESOLVED_CACHE):
    if not p.exists():
        sys.exit(f"missing cache: {p}")
print("caches found ✓")

caches found ✓


In [50]:
resolved_data         = json.loads(RESOLVED_CACHE.read_text())
mention_to_canonical: dict[str, str] = resolved_data["mention_to_canonical"]
print(f"resolved entities : {resolved_data['total_entities']:>5}")
print(f"surface forms     : {len(mention_to_canonical):>5}")

entity_cache = json.loads(ENTITY_CACHE.read_text())
mentions     = entity_cache["mentions"]

_doc_cache: dict[str, str] = {}

def _doc_text(source_file: str) -> str:
    if source_file not in _doc_cache:
        p = Path(source_file)
        _doc_cache[source_file] = p.read_text(encoding="utf-8", errors="ignore") if p.exists() else ""
    return _doc_cache[source_file]


ChunkKey = tuple[str, str]
chunks: dict[ChunkKey, dict] = {}

for m in mentions:
    key: ChunkKey = (m["source_file"], str(m["episode_idx"]))
    if key in chunks:
        continue
    episode = m["episode"]
    doc     = _doc_text(m["source_file"])
    offset  = doc.find(episode)
    chunks[key] = {
        "doc_id":      m["source_file"],
        "episode_idx": str(m["episode_idx"]),
        "text":        episode,
        "char_start":  offset,
        "char_end":    offset + len(episode) if offset >= 0 else -1,
    }

chunk_entities: dict[ChunkKey, list[str]] = {k: [] for k in chunks}
for m in mentions:
    key = (m["source_file"], str(m["episode_idx"]))
    canon = mention_to_canonical.get(m["text"])
    if canon and canon not in chunk_entities[key]:
        chunk_entities[key].append(canon)

bad = sum(1 for c in chunks.values() if c["char_start"] < 0)
print(f"\nchunks            : {len(chunks):>5}")
print(f"bad char offsets  : {bad:>5}")

resolved entities :  1026
surface forms     :   880

chunks            :   182
bad char offsets  :    62


## RelationEdge model

`relation_description` is the atomic fact — a single sentence copied or lightly paraphrased
from the source that explains the connection. Both entity endpoints are stored; the object
may be unresolved if the object entity was not extracted upstream.

In [53]:
%history -g unresolved_objects

266/6:
FACT_TYPE_VALUES = [
    "assertion",
    "definition",
    "opinion",
    "instruction",
    "metaphor",
    "prediction",
    "other_non_fact",
]


class RelationEdge(BaseModel):
    """Typed entity-to-entity edge. Keyed by canonical IDs on both ends.

    These are PPR-walkable edges in the relation layer.
    They are NOT reified as HAS_FACT attribute nodes.
    """

    relation_id:        str   # stable hash of (doc_id, char_start, relation_type, subject, object_raw)

    # endpoints — both must be canonical
    subject_entity_id:  str   # canonical, from the entity list
    relation_type:      str   # usesTool | worksFor | hasPart | …
    relation_type_raw:  str   = ""   # when relation_type="other"
    object_raw:         str   # verbatim object name from source
    object_entity_id:   str | None = None   # canonical, resolved via mention_to_canonical
    object_resolved:    bool  = False

    # fact classification (same gate as fact pipeline)
    fact_type:          str

In [55]:
import re, json, hashlib
from typing import NamedTuple
from collections import defaultdict
from pydantic import BaseModel

FACT_TYPE_VALUES = [
    "assertion",       # checkable claim — edge if object resolves, node-fact otherwise
    "definition",      # copular/attributive — edge if entity object, node-fact if class object
    "opinion",         # personal-notes corpus: KEPT as node-fact, labeled
    "instruction",     # KEPT as node-fact, labeled
    "prediction",      # KEPT as node-fact, labeled
    "metaphor",        # log-only
    "other_non_fact",  # log-only
]

# fact types that carry retrievable content (edge or node-fact)
_CONTENT_TYPES = {"assertion", "definition", "opinion", "instruction", "prediction"}
# fact types eligible to become PPR edges (when object resolves)
_EDGE_TYPES = {"assertion", "definition"}


class RelationEdge(BaseModel):
    """Entity-to-entity RELATES_TO edge with a plain-English description."""
    relation_id:          str
    subject_entity_id:    str
    relation_description: str
    object_raw:           str
    object_entity_id:     str | None = None
    object_resolved:      bool        = False
    fact_type:            str         = "assertion"
    valid_from:           str | None  = None
    valid_to:             str | None  = None
    temporal_inferred:    bool        = False
    doc_id:               str
    char_start:           int
    char_end:             int
    source_quote:         str
    quote_verified:       bool        = False
    span_verified:        bool        = False
    confidence:           float       = 1.0
    extractor_version:    str         = EXTRACTOR_VERSION


class NodeFact(BaseModel):
    """Fact about ONE entity. Indexed for asymmetric fact ranking; creates NO
    edge. Sources: objectless statements, unresolved objects, class-valued
    definitions, and opinions/instructions/predictions (labeled by fact_type)."""
    fact_id:            str
    entity_id:          str
    fact_description:   str
    fact_type:          str   = "assertion"
    doc_id:             str
    char_start:         int
    char_end:           int
    source_quote:       str
    quote_verified:     bool  = False
    confidence:         float = 1.0
    provenance:         str   = "extraction"
    extractor_version:  str   = EXTRACTOR_VERSION


class ChunkResult(NamedTuple):
    edges:              list[RelationEdge]
    node_facts:         list[NodeFact]
    non_facts:          list[dict]     # metaphor / other_non_fact only
    rejected_subjects:  list[dict]
    uncovered:          list[str]


def make_relation_id(doc_id: str, char_start: int, subject: str,
                     object_raw: str, description: str) -> str:
    desc_h = hashlib.sha1(description.encode()).hexdigest()[:8]
    key = f"{doc_id}|{char_start}|{subject}|{object_raw}|{desc_h}"
    return hashlib.sha1(key.encode()).hexdigest()[:16]


def make_fact_id(doc_id: str, char_start: int, entity: str, description: str) -> str:
    desc_h = hashlib.sha1(description.encode()).hexdigest()[:8]
    key = f"{doc_id}|{char_start}|{entity}|{desc_h}"
    return hashlib.sha1(key.encode()).hexdigest()[:16]


# ── alias map (from resolution output) ─────────────────────────────────────────
canonical_aliases: dict[str, list[str]] = defaultdict(list)
for _surface, _canon in mention_to_canonical.items():
    if _surface != _canon:
        canonical_aliases[_canon].append(_surface)

_mention_lower_map: dict[str, str] = {}
for _surface, _cid in mention_to_canonical.items():
    _mention_lower_map.setdefault(_surface.lower(), _cid)


def resolve_to_canonical(name: str) -> tuple[str | None, bool]:
    """surface or canonical name -> (canonical_id, resolved). Dictionary replay
    of resolution decisions only — no fuzzy matching."""
    if not name:
        return None, False
    canon = mention_to_canonical.get(name)
    if canon:
        return canon, True
    canon = _mention_lower_map.get(name.lower())
    if canon:
        return canon, True
    return None, False


def format_entities_with_aliases(entities: list[str], max_aliases: int = 4) -> str:
    lines = []
    for e in entities:
        aliases = canonical_aliases.get(e, [])[:max_aliases]
        tag = f"  (in text also as: {', '.join(aliases)})" if aliases else ""
        lines.append(f"- {e}{tag}")
    return "\n".join(lines)


def entities_for_chunk(chunk_text: str, all_canonicals: list[str]) -> list[str]:
    """Chunk-scoped entity list: canonicals with ANY alias present in the chunk.
    Use this to build the per-chunk `entities` argument if you currently pass a
    document- or corpus-level list."""
    text_lower = chunk_text.lower()
    hits = []
    for canon in all_canonicals:
        surfaces = [canon] + canonical_aliases.get(canon, [])
        if any(s.lower() in text_lower for s in surfaces):
            hits.append(canon)
    return hits

print("models + alias map loaded ✓")


# ── prompt ─────────────────────────────────────────────────────────────────────
_SYSTEM_PROMPT = """\
You read a text chunk and extract atomic facts about the given entities for a personal knowledge graph. Each fact is one sentence about a subject entity, optionally connecting it to an object entity.

CLASSIFY every statement the text makes about a listed entity — do not silently skip anything except pure scalars:
- assertion: a checkable claim about the subject. ("Alice works for Acme Corp as a data scientist.")
- definition: a copular or attributive statement saying what the subject IS — its class, role, or nature. ("Neo4j is a graph database.")
- opinion: a judgment or stance. ("Direct Mentorship is not really scalable.") These are the author's notes — emit them, they are valuable content.
- instruction: guidance or a rule. ("Never talk about the idea in the abstract.") Emit these too.
- prediction: a claim about the future. Emit.
- metaphor / other_non_fact: emit with the label so the pipeline can log them.

SKIP entirely (a separate stage extracts these): pure scalar attributes — headcounts, dates as standalone facts, money amounts, percentages, durations. "The team has 12 people" → skip. A date ANCHORING a fact goes in valid_from/valid_to, never in the description.

OBJECT RULES — the most important part:
- raw_object_str must be an ENTITY MENTION: a noun phrase naming a specific thing, person, organization, method, product, or concept — something that could itself be a node in the graph. It is NEVER a verb phrase, a clause, or the remainder of the sentence. Test before writing it: could this string plausibly be an item in the ENTITIES list?
- If a statement asserts something about the subject but has NO entity object — a property, behavior, or state ("Medoid is interpretable", "Saved data is messy") — set raw_object_str to "" and still emit the item. It becomes a fact attached to the subject alone.
- When the text supports it, prefer objects that appear in the ENTITIES list (under their canonical name or a listed alias). Copy object names VERBATIM as they appear in the text.

COVERAGE REQUIREMENT: every entity in ENTITIES must appear as subject or object of at least one emitted item, OR be listed in uncovered_entities. Entities may appear in the text under any of their listed aliases — check aliases before declaring one uncovered. Check definitions and appositions, not only verbs. An entity mentioned purely in passing, with nothing asserted about it, belongs in uncovered_entities; that is a valid outcome.

RULES for each item:
- relation_description: ONE sentence, one claim, grounded in the text. Copy or lightly paraphrase; never infer beyond what is stated. Use the subject's canonical name, not a pronoun or alias.
- Direction: the agent/possessor/part-owner is the subject. Emit each fact once — never both A→B and B→A.
- source_quote: the VERBATIM sentence from the chunk supporting this item, character-for-character. Two adjacent sentences if support spans both.
- subject_entity_id: copied VERBATIM from the ENTITIES list (canonical form, not an alias).
- valid_from / valid_to: ISO format, only if explicitly stated; partial dates for partial information ("March 2024" → "2024-03"); else null. temporal_inferred = true only if you derived a period not literally present.

You will receive TEXT (the chunk) and ENTITIES (canonical entity IDs with their known text aliases)."""


_SCHEMA = {
    "type": "object",
    "additionalProperties": False,
    "properties": {
        "relations": {
            "type": "array",
            "items": {
                "type": "object",
                "additionalProperties": False,
                "properties": {
                    "subject_entity_id":    {"type": "string"},
                    "raw_object_str":       {"type": "string"},
                    "relation_description": {"type": "string"},
                    "fact_type":            {"type": "string", "enum": FACT_TYPE_VALUES},
                    "source_quote":         {"type": "string"},
                    "valid_from":           {"type": ["string", "null"]},
                    "valid_to":             {"type": ["string", "null"]},
                    "temporal_inferred":    {"type": "boolean"},
                },
                "required": [
                    "subject_entity_id", "raw_object_str", "relation_description",
                    "fact_type", "source_quote", "valid_from", "valid_to",
                    "temporal_inferred",
                ],
            },
        },
        "uncovered_entities": {"type": "array", "items": {"type": "string"}},
    },
    "required": ["relations", "uncovered_entities"],
}


def extract_relations_from_chunk(chunk: dict, entities: list[str]) -> dict:
    if not entities:
        return {"relations": [], "uncovered_entities": []}
    user_msg = (
        f"TEXT:\n{chunk['text']}\n\n"
        "ENTITIES (canonical IDs — copy exactly as subject_entity_id; "
        "aliases show how they may appear in the text):\n"
        + format_entities_with_aliases(entities)
    )
    resp = client.with_options(timeout=90.0, max_retries=2).chat.completions.create(
        model=EXTRACT_MODEL,
        temperature=0,
        messages=[
            {"role": "system", "content": _SYSTEM_PROMPT},
            {"role": "user",   "content": user_msg},
        ],
        response_format={
            "type": "json_schema",
            "json_schema": {"name": "relations", "strict": True, "schema": _SCHEMA},
        },
    )
    return json.loads(resp.choices[0].message.content)

print("Stage 1 ready ✓  (object constraint, alias-aware coverage)")


# ── verification + confidence ──────────────────────────────────────────────────
_DIGITS_RE = re.compile(r"\d+")

def _norm_ws(s: str) -> str:
    return re.sub(r"\s+", " ", s).strip()

def verify_quote_and_span(raw_object_str, model_quote, chunk_text,
                          valid_from, valid_to) -> dict:
    quote_verified = bool(model_quote) and _norm_ws(model_quote) in _norm_ws(chunk_text)
    haystack = model_quote if quote_verified else chunk_text
    # objectless items pass span check vacuously
    span_verified = (not raw_object_str) or raw_object_str.lower() in haystack.lower()
    temporal_inferred = False
    for date_val in filter(None, [valid_from, valid_to]):
        digits = _DIGITS_RE.findall(date_val)
        if digits and not all(d in haystack for d in digits):
            temporal_inferred = True
            break
    return {"quote_verified": quote_verified, "span_verified": span_verified,
            "temporal_inferred": temporal_inferred}

def score_confidence(object_resolved, quote_verified, span_verified,
                     temporal_inferred, has_object=True) -> float:
    score = 1.00
    if has_object and not object_resolved: score -= 0.20
    if not quote_verified:                 score -= 0.15
    if not span_verified:                  score -= 0.30
    if temporal_inferred:                  score -= 0.10
    return round(max(0.0, score), 2)

print("Stage 3 verify + confidence ready ✓")


# ── assembly ───────────────────────────────────────────────────────────────────
def process_chunk(chunk: dict, entities: list[str]) -> ChunkResult:
    raw = extract_relations_from_chunk(chunk, entities)
    edges:             list[RelationEdge] = []
    node_facts:        list[NodeFact]     = []
    non_facts:         list[dict]         = []
    rejected_subjects: list[dict]         = []
    entity_set = set(entities)

    for rr in raw["relations"]:
        fact_type = rr.get("fact_type", "assertion")
        raw_obj   = rr["raw_object_str"].strip()
        desc      = rr["relation_description"]
        quote     = rr.get("source_quote", "")

        # metaphor / other_non_fact → log only
        if fact_type not in _CONTENT_TYPES:
            non_facts.append({
                "fact_type": fact_type, "subject_entity_id": rr["subject_entity_id"],
                "raw_object_str": raw_obj, "relation_description": desc,
                "source_quote": quote, "doc_id": chunk["doc_id"],
            })
            continue

        # subject: exact, else alias fallback, else reject
        subj = rr["subject_entity_id"]
        if subj not in entity_set:
            subj_canon, ok = resolve_to_canonical(subj)
            if ok and subj_canon in entity_set:
                subj = subj_canon
            else:
                rejected_subjects.append({
                    "subject_attempted": rr["subject_entity_id"],
                    "raw_object_str": raw_obj, "relation_description": desc,
                    "doc_id": chunk["doc_id"],
                })
                continue

        obj_canon, obj_resolved = resolve_to_canonical(raw_obj)
        has_object = bool(raw_obj)
        verify = verify_quote_and_span(raw_obj, quote, chunk["text"],
                                       rr.get("valid_from"), rr.get("valid_to"))
        temporal_inf = verify["temporal_inferred"] or rr.get("temporal_inferred", False)
        conf = score_confidence(obj_resolved, verify["quote_verified"],
                                verify["span_verified"], temporal_inf, has_object)

        # edge only if: edge-eligible type AND object resolved to a node
        if fact_type in _EDGE_TYPES and obj_resolved:
            edges.append(RelationEdge(
                relation_id=make_relation_id(
                    chunk["doc_id"], chunk["char_start"], subj, raw_obj, desc),
                subject_entity_id=subj,
                relation_description=desc,
                object_raw=raw_obj,
                object_entity_id=obj_canon,
                object_resolved=True,
                fact_type=fact_type,
                valid_from=rr.get("valid_from"),
                valid_to=rr.get("valid_to"),
                temporal_inferred=temporal_inf,
                doc_id=chunk["doc_id"],
                char_start=chunk["char_start"],
                char_end=chunk["char_end"],
                source_quote=quote,
                quote_verified=verify["quote_verified"],
                span_verified=verify["span_verified"],
                confidence=conf,
            ))
        else:
            # everything else with content → node-fact on the subject
            if has_object and not obj_resolved:
                prov = "unresolved_object"
            elif not has_object:
                prov = "objectless"
            else:
                prov = fact_type          # opinion/instruction/prediction w/ resolved obj
            node_facts.append(NodeFact(
                fact_id=make_fact_id(chunk["doc_id"], chunk["char_start"], subj, desc),
                entity_id=subj,
                fact_description=desc,
                fact_type=fact_type,
                doc_id=chunk["doc_id"],
                char_start=chunk["char_start"],
                char_end=chunk["char_end"],
                source_quote=quote,
                quote_verified=verify["quote_verified"],
                confidence=conf,
                provenance=prov,
            ))

    uncovered = [e for e in raw.get("uncovered_entities", []) if e in entity_set]
    touched  = {e.subject_entity_id for e in edges}
    touched |= {e.object_entity_id for e in edges if e.object_entity_id}
    touched |= {f.entity_id for f in node_facts}
    uncovered = sorted((entity_set - touched) | set(uncovered) - touched)

    return ChunkResult(edges=edges, node_facts=node_facts, non_facts=non_facts,
                       rejected_subjects=rejected_subjects, uncovered=uncovered)

print("pipeline assembled ✓")
print("  edges → RELATES_TO (both endpoints resolved, assertion/definition only)")
print("  node_facts → per-entity fact index; provenance ∈ "
      "{objectless, unresolved_object, opinion, instruction, prediction, definition}")
print("  uncovered → residual-singleton metric / mention-context fallback input")

models + alias map loaded ✓
Stage 1 ready ✓  (object constraint, alias-aware coverage)
Stage 3 verify + confidence ready ✓
pipeline assembled ✓
  edges → RELATES_TO (both endpoints resolved, assertion/definition only)
  node_facts → per-entity fact index; provenance ∈ {objectless, unresolved_object, opinion, instruction, prediction, definition}
  uncovered → residual-singleton metric / mention-context fallback input


## Demo run — 10 chunks

In [60]:
from collections import Counter
from pathlib import Path

DEMO_CHUNKS = 5
demo_edges:      list[RelationEdge] = []
demo_node_facts: list[NodeFact]     = []
demo_non_facts:  list[dict]         = []
demo_rejected:   list[dict]         = []
demo_uncovered:  dict[str, list[str]] = {}

chunk_items = [
    (key, chunk) for key, chunk in chunks.items()
    if chunk["char_start"] >= 0 and chunk_entities.get(key)
][:DEMO_CHUNKS]

for key, chunk in chunk_items:
    try:
        result = process_chunk(chunk, chunk_entities[key])
        demo_edges.extend(result.edges)
        demo_node_facts.extend(result.node_facts)
        demo_non_facts.extend(result.non_facts)
        demo_rejected.extend(result.rejected_subjects)
        if result.uncovered:
            demo_uncovered[str(key)] = result.uncovered
    except (AttributeError, KeyError, NameError):
        raise                                   # code bug — crash loudly
    except Exception as e:
        print(f"  chunk {key} failed: {e!r}")

total = len(demo_edges) + len(demo_node_facts) + len(demo_non_facts) + len(demo_rejected)
n_uncov = sum(len(v) for v in demo_uncovered.values())

print("=" * 64)
print(f"  RELATION EXTRACTION  ({DEMO_CHUNKS} chunks)")
print("=" * 64)
print(f"  total LLM claims    : {total}")
print(f"  edges (2 endpoints) : {len(demo_edges)}")
print(f"  node-facts          : {len(demo_node_facts)}")
for prov, cnt in sorted(Counter(f.provenance for f in demo_node_facts).items(), key=lambda x: -x[1]):
    print(f"    {prov:<22} {cnt}")
print(f"  non-facts (logged)  : {len(demo_non_facts)}")
for ft, cnt in sorted(Counter(n['fact_type'] for n in demo_non_facts).items(), key=lambda x: -x[1]):
    print(f"    {ft:<22} {cnt}")
print(f"  invalid subjects    : {len(demo_rejected)}")
print(f"  uncovered entities  : {n_uncov}")
print()

if demo_edges:
    confs = sorted(e.confidence for e in demo_edges)
    buckets = Counter(round(c, 1) for c in confs)
    qv = sum(e.quote_verified for e in demo_edges)
    print(f"  CONFIDENCE  min={confs[0]:.2f}  max={confs[-1]:.2f}  mean={sum(confs)/len(confs):.2f}")
    for score in sorted(buckets):
        print(f"    {score:.1f}  {'█' * buckets[score]} ({buckets[score]})")
    print(f"  quote verified      : {qv}/{len(demo_edges)}")
    print()

print("=" * 64)
print(f"  ALL EDGES ({len(demo_edges)})")
print("=" * 64)
for e in sorted(demo_edges, key=lambda x: -x.confidence):
    qv_mark = "✓" if e.quote_verified else "✗"
    sv_mark = "✓" if e.span_verified  else "✗"
    print(f"  [{e.confidence:.2f}] quote={qv_mark} span={sv_mark} type={e.fact_type}")
    print(f"    {e.subject_entity_id}")
    print(f"    -RELATES_TO-> {e.object_raw!r}  [{e.object_entity_id}]")
    print(f"    '{e.relation_description}'")
    print(f"    {Path(e.doc_id).name}  [{e.char_start}:{e.char_end}]")
    print()

print("=" * 64)
print(f"  SAMPLE NODE-FACTS ({min(10, len(demo_node_facts))} of {len(demo_node_facts)})")
print("=" * 64)
for f in sorted(demo_node_facts, key=lambda x: -x.confidence)[:10]:
    print(f"  [{f.confidence:.2f}] {f.provenance:<18} {f.entity_id}")
    print(f"    '{f.fact_description}'")
    print()

if demo_uncovered:
    print("=" * 64)
    print("  UNCOVERED BY CHUNK")
    print("=" * 64)
    for k, ents in demo_uncovered.items():
        print(f"  {k}: {ents}")

  RELATION EXTRACTION  (5 chunks)
  total LLM claims    : 27
  edges (2 endpoints) : 12
  node-facts          : 14
    objectless             10
    instruction            3
    opinion                1
  non-facts (logged)  : 1
    metaphor               1
  invalid subjects    : 0
  uncovered entities  : 24

  CONFIDENCE  min=1.00  max=1.00  mean=1.00
    1.0  ████████████ (12)
  quote verified      : 12/12

  ALL EDGES (12)
  [1.00] quote=✓ span=✓ type=assertion
    GraphRAG
    -RELATES_TO-> 'componets'  [componets]
    'GraphRAG has componets'
    Construction.md  [0:366]

  [1.00] quote=✓ span=✓ type=assertion
    LLM
    -RELATES_TO-> 'triplets'  [triplets]
    'LLM generates triplets'
    Construction.md  [0:366]

  [1.00] quote=✓ span=✓ type=assertion
    Dual-graph / Hybrid index
    -RELATES_TO-> 'Entity relation KG'  [Entity relation KG]
    'Dual-graph / Hybrid index has two separate graphs including Entity relation KG'
    Construction.md  [0:366]

  [1.00] quote=✓ span=✓

## Bulk run — all chunks → `relations_cache.json`

In [64]:
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.notebook import tqdm as tqdm_nb
from collections import Counter
import json

BULK_WORKERS = 8
SAVE_EVERY   = 50

eligible = [
    (key, chunk) for key, chunk in chunks.items()
    if chunk["char_start"] >= 0 and chunk_entities.get(key)
]
print(f"{len(eligible)} eligible chunks")

all_edges:      list[dict] = []
all_node_facts: list[dict] = []
all_non_facts:  list[dict] = []
all_rejected:   list[dict] = []
all_uncovered:  dict[str, list[str]] = {}
failed_chunks:  list[str] = []


def _process(args: tuple):
    key, chunk = args
    result = process_chunk(chunk, chunk_entities[key])
    return (
        str(key),
        [e.model_dump(mode="json") for e in result.edges],
        [f.model_dump(mode="json") for f in result.node_facts],
        result.non_facts,
        result.rejected_subjects,
        result.uncovered,
    )


def _save() -> None:
    RELATIONS_CACHE.write_text(json.dumps({
        "edges":              all_edges,
        "node_facts":         all_node_facts,
        "non_facts":          all_non_facts,
        "rejected_subjects":  all_rejected,
        "uncovered_by_chunk": all_uncovered,
        "failed_chunks":      failed_chunks,
    }, ensure_ascii=False, indent=1))


with ThreadPoolExecutor(max_workers=BULK_WORKERS) as executor:
    futures = {executor.submit(_process, item): item for item in eligible}
    with tqdm_nb(total=len(eligible), desc="relation extraction") as bar:
        for i, future in enumerate(as_completed(futures)):
            key, _ = futures[future]
            try:
                k, eb, nfb, xfb, rjb, unc = future.result()
                all_edges.extend(eb)
                all_node_facts.extend(nfb)
                all_non_facts.extend(xfb)
                all_rejected.extend(rjb)
                if unc:
                    all_uncovered[k] = unc
            except (AttributeError, KeyError, NameError, TypeError):
                raise                                   # code bug — fix, don't skip
            except Exception as e:
                print(f"  ✗ chunk {key} failed: {e!r}")
                failed_chunks.append(str(key))
            bar.update(1)
            if (i + 1) % SAVE_EVERY == 0:
                _save()

_save()

# ── summary ────────────────────────────────────────────────────────────────────
n_uncov   = sum(len(v) for v in all_uncovered.values())
conf_vals = [e["confidence"] for e in all_edges]
qv        = sum(1 for e in all_edges if e.get("quote_verified"))

print(f"\n{len(all_edges)} edges | {len(all_node_facts)} node-facts | "
      f"{len(all_non_facts)} non-facts | {len(all_rejected)} rejected subjects | "
      f"{n_uncov} uncovered | {len(failed_chunks)} failed chunks")
if all_edges:
    print(f"  quote verified  : {qv}/{len(all_edges)} ({100*qv//len(all_edges)}%)")
    print(f"  avg confidence  : {sum(conf_vals)/len(conf_vals):.2f}")
print(f"  node-fact provenance:")
for prov, cnt in sorted(Counter(f['provenance'] for f in all_node_facts).items(), key=lambda x: -x[1]):
    print(f"    {prov:<22} {cnt}")
print(f"  saved to        : {RELATIONS_CACHE}")

86 eligible chunks


relation extraction:   0%|          | 0/86 [00:00<?, ?it/s]


248 edges | 294 node-facts | 1 non-facts | 30 rejected subjects | 207 uncovered | 0 failed chunks
  quote verified  : 210/248 (84%)
  avg confidence  : 0.97
  node-fact provenance:
    objectless             214
    unresolved_object      58
    instruction            18
    opinion                4
  saved to        : relations_cache.json


In [65]:
edges = cache["edges"] if isinstance(cache, dict) else cache
unres = [e for e in edges if not e.get("object_resolved")]
sample = random.sample(unres, min(30, len(unres)))
for u in sample:
    print(f"{u.get('object_raw', u.get('raw_object_str'))!r:45} ← {u['relation_description'][:70]}")
print(cache)

NameError: name 'cache' is not defined

In [ ]:
import json

cache = json.load(open("relations_cache.json"))
edges = cache["edges"]

kept_edges, salvaged_facts = [], []
for e in edges:
    if e.get("object_resolved"):
        kept_edges.append(e)
    else:
        salvaged_facts.append({
            "fact_id":          e["relation_id"],
            "entity_id":        e["subject_entity_id"],
            "fact_description": e["relation_description"],
            "doc_id":           e["doc_id"],
            "char_start":       e["char_start"],
            "char_end":         e["char_end"],
            "source_quote":     e.get("source_quote", ""),
            "quote_verified":   e.get("quote_verified", False),
            "confidence":       e["confidence"],
            "provenance":       "salvaged_unresolved_edge",
        })

print(f"kept {len(kept_edges)} true edges, salvaged {len(salvaged_facts)} node-facts")
cache["edges"] = kept_edges
cache["node_facts"] = cache.get("node_facts", []) + salvaged_facts
json.dump(cache, open("relations_cache_v2.json", "w"), ensure_ascii=False, indent=1)

## Stage 4 — Embed `relation_description` (asymmetric retrieval)

Entity mentions in `entity_extraction.ipynb` are embedded *symmetrically* — every
mention is encoded the same way and clustered against every other mention. Facts are
different: they are retrieved by a user's natural-language question, so the query and
the fact live in different distributions. We use `intfloat/e5-base-v2` for this —
an asymmetric embedding model where **both** sides need a prefix: `"query: "` on
questions, `"passage: "` on node/fact text. Unlike some asymmetric models where only
the query gets a prefix, e5 requires a prefix on every input — omitting `"passage: "`
on facts is the most common silent bug with this model (embeddings still "work", they're
just miscalibrated).

- Facts (`relation_description`) are embedded once, in bulk, and cached to `relation_vectors.npz`.
- At query time, embed the question with `is_query=True` and take the dot product
  against the cached fact matrix (vectors are L2-normalized, so dot product == cosine similarity).

In [23]:
import numpy as np
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModel

EMBED_MODEL     = "intfloat/e5-base-v2"
QUERY_PREFIX    = "query: "
PASSAGE_PREFIX  = "passage: "
FACT_MAX_TOKENS = 512

torch.set_num_threads(1)
_fact_tokenizer = AutoTokenizer.from_pretrained(EMBED_MODEL)
_fact_model     = AutoModel.from_pretrained(EMBED_MODEL)
_fact_model.eval()


def _mean_pool(last_hidden_state: torch.Tensor, attention_mask: torch.Tensor) -> torch.Tensor:
    """Mean-pool token embeddings over real (non-padding) tokens."""
    mask   = attention_mask.unsqueeze(-1).float()
    summed = (last_hidden_state * mask).sum(dim=1)
    counts = mask.sum(dim=1).clamp(min=1e-9)
    return summed / counts


def embed_facts(
    texts: list[str], is_query: bool = False, batch_size: int = 32
) -> list[list[float]]:
    """Embed sentences with e5-base-v2's asymmetric convention.

    e5 requires a prefix on EVERY input, not just queries:
    "query: " for questions, "passage: " for facts/nodes. A missing "passage: "
    prefix is the most common silent bug — embeddings still compute, they're
    just off-distribution and retrieval quality quietly degrades.
    Output vectors are L2-normalized, so cosine similarity is a plain dot product.
    """
    prefix = QUERY_PREFIX if is_query else PASSAGE_PREFIX
    texts  = [prefix + t for t in texts]

    vectors: list[list[float]] = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i : i + batch_size]
        encoding = _fact_tokenizer(
            batch, return_tensors="pt", truncation=True,
            padding=True, max_length=FACT_MAX_TOKENS,
        )
        with torch.no_grad():
            output = _fact_model(**encoding)
        pooled = _mean_pool(output.last_hidden_state, encoding["attention_mask"])
        normed = F.normalize(pooled, p=2, dim=1)
        vectors.extend(normed.numpy().tolist())
    return vectors


print(f"loaded {EMBED_MODEL} for fact embedding (asymmetric: 'query: ' vs 'passage: ' prefixes, both required)")

ImportError: tokenizers>=0.19,<0.20 is required for a normal functioning of this module, but found tokenizers==0.23.1.
Try: `pip install transformers -U` or `pip install -e '.[dev]'` if you're working with git main

In [ ]:
RELATION_VECTOR_CACHE = Path("relation_vectors.npz")

relations_data = json.loads(RELATIONS_CACHE.read_text())
relation_ids   = [e["relation_id"] for e in relations_data]
descriptions   = [e["relation_description"] for e in relations_data]

fact_vectors = embed_facts(descriptions, is_query=False)
fact_matrix  = np.array(fact_vectors, dtype=np.float32)

np.savez(
    RELATION_VECTOR_CACHE,
    relation_ids=np.array(relation_ids),
    matrix=fact_matrix,
)
print(f"{len(relation_ids)} fact vectors saved -> {RELATION_VECTOR_CACHE}")

### Retrieval demo

Sanity check: embed a question with the query prefix and retrieve the closest
`relation_description` facts by dot product. `demo_entity` is picked from the actual
extracted data (the most frequent subject) so the query is grounded in real content,
not invented.

In [ ]:
from collections import Counter

by_relation_id = {e["relation_id"]: e for e in relations_data}


def search_facts(query: str, top_k: int = 5) -> list[tuple[float, dict]]:
    """Dot-product search over cached fact vectors for a natural-language query."""
    query_vec = np.array(embed_facts([query], is_query=True)[0], dtype=np.float32)
    scores    = fact_matrix @ query_vec
    top_idx   = np.argsort(-scores)[:top_k]
    return [(float(scores[i]), by_relation_id[relation_ids[i]]) for i in top_idx]


subject_counts = Counter(e["subject_entity_id"] for e in relations_data)

if subject_counts:
    demo_entity = subject_counts.most_common(1)[0][0]
    demo_query  = f"What do we know about {demo_entity}?"
    print(f"query: {demo_query!r}\n")
    for score, edge in search_facts(demo_query):
        print(f"  [{score:.3f}] {edge['subject_entity_id']} -RELATES_TO-> "
              f"{edge.get('object_entity_id') or edge['object_raw']}")
        print(f"          {edge['relation_description']}")
else:
    print("no relations yet - run the bulk extraction cell above first")

## Graph write — next step

Every edge writes as `:RELATES_TO`. The `relation_description` property holds the
plain-English sentence and is what you display, search, or embed for similarity.

```cypher
MATCH (s:Entity {canonical_id: $subject_entity_id})
MATCH (o:Entity {canonical_id: $object_entity_id})
MERGE (s)-[r:RELATES_TO {relation_id: $relation_id}]->(o)
SET r.relation_description = $relation_description,
    r.source_quote         = $source_quote,
    r.doc_id               = $doc_id,
    r.confidence           = $confidence,
    r.valid_from           = $valid_from,
    r.valid_to             = $valid_to
```

Skip edges where `object_entity_id` is null — no target node to connect to.

**PPR note:** PPR walks `:RELATES_TO` edges regardless of description.
To filter by semantic type later, embed `relation_description` vectors and
cluster at query time rather than locking in a label at extraction time.